# 02 — vLLM serving + the OpenAI client

vLLM exposes an **OpenAI-compatible HTTP server** (`vllm serve`). Serving once and hitting
it over HTTP lets many consumers share a single engine init — the basis for adapter sweeps
(serve the base + all LoRA adapters, request each by name).

**Library focus:** `vllm serve` (CLI/subprocess), the **`openai`** Python client.

> This notebook launches a real server subprocess and tears it down at the end. Equivalent
> one-liner in a terminal: `vllm serve Qwen/Qwen3.5-4B --host 0.0.0.0 --port 8000`.

In [ ]:
MODEL = "Qwen/Qwen3.5-4B"
HOST, PORT = "127.0.0.1", 8000
BASE_URL = f"http://{HOST}:{PORT}"

## Launch the server and wait for `/health`

Spawn `vllm serve` in its own session so we can signal the whole process group on teardown
(vLLM spawns worker processes). Poll `/health` until it returns 200.

In [ ]:
import os
import signal
import subprocess
import time
import urllib.request

cmd = [
    "vllm",
    "serve",
    MODEL,
    "--host",
    HOST,
    "--port",
    str(PORT),
    "--gpu-memory-utilization",
    "0.9",
]
print("$", " ".join(cmd))
proc = subprocess.Popen(cmd, start_new_session=True)


def healthy(url, timeout=5.0):
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r:
            return r.status == 200
    except Exception:
        return False


deadline = time.monotonic() + 600
while time.monotonic() < deadline:
    if proc.poll() is not None:
        raise RuntimeError(f"vllm serve exited early (code {proc.returncode})")
    if healthy(f"{BASE_URL}/health"):
        print("server healthy at", BASE_URL)
        break
    time.sleep(3)

## Call it with the `openai` client

Point the OpenAI SDK at `{BASE_URL}/v1` with a dummy key. Both the chat and raw-completions
endpoints work; `seed` makes a request reproducible.

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=f"{BASE_URL}/v1", api_key="EMPTY")
print("served model:", client.models.list().data[0].id)

chat = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "One fun fact about octopuses."}],
    temperature=0.7,
    max_tokens=80,
    seed=0,
)
print("chat:", chat.choices[0].message.content.strip())

comp = client.completions.create(
    model=MODEL,
    prompt="The mitochondrion is",
    temperature=0.8,
    max_tokens=40,
    seed=0,
)
print("completion:", comp.choices[0].text.strip())

## Serving LoRA adapters by name

Add `--enable-lora --lora-modules <name>=<path>` and request each adapter via the `model`
field. One engine, many adapters — the pattern the Exp-0 sweep uses to evaluate a whole
shortlist off a single init. (Shown as the command; uncomment after training an adapter in
notebook 05.)

```sh
vllm serve Qwen/Qwen3.5-4B --enable-lora --max-lora-rank 16 \
    --lora-modules med=runs/train_xxx/adapter
# then:  client.completions.create(model="med", prompt=..., ...)
```

## Tear down

Kill the whole process group so no server is orphaned.

In [ ]:
try:
    os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
    proc.wait(timeout=20)
except (ProcessLookupError, subprocess.TimeoutExpired):
    os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
print("server stopped")

> **Project pointer:** `llm_core.serving.ServedVLLM` is a context manager that builds this
> command, polls `/health`, and tears the group down on exit;
> `llm_core.generation.served.generate_served` is the client-side sampling helper. lm-eval
> can target the same server via its `local-completions` backend (notebook 07).